In [20]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.applications import EfficientNetB0

In [21]:
DATASET_PATH = "../dataset/disease"

IMG_SIZE = (224,224)

BATCH_SIZE = 16

In [22]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print(class_names)

Found 5452 files belonging to 4 classes.
Using 4362 files for training.
Found 5452 files belonging to 4 classes.
Using 1090 files for validation.
['Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_healthy']


In [23]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

In [24]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

In [25]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

In [26]:
inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    4,
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,383,655 (16.72 MB)

 Trainable params: 331,524 (1.26 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [27]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "../models/best_efficientnet.keras",
        save_best_only=True
    )
]

In [29]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 136s 476ms/step - accuracy: 0.8505 - loss: 0.4432 - val_accuracy: 0.9505 - val_loss: 0.1476 - learning_rate: 0.0010
Epoch 2/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 103s 378ms/step - accuracy: 0.9195 - loss: 0.2482 - val_accuracy: 0.9578 - val_loss: 0.1375 - learning_rate: 0.0010
Epoch 3/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 104s 381ms/step - accuracy: 0.9328 - loss: 0.2139 - val_accuracy: 0.9624 - val_loss: 0.1004 - learning_rate: 0.0010
Epoch 4/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 104s 382ms/step - accuracy: 0.9395 - loss: 0.1843 - val_accuracy: 0.9606 - val_loss: 0.1036 - learning_rate: 0.0010
Epoch 5/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.9468 - loss: 0.1732
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
273/273 ━━━━━━━━━━━━━━━━━━━━ 103s 377ms/step - accuracy: 0.9498 - loss: 0.1590 - val_accuracy: 0.9670 - val_loss: 0.1034 - learning_rate: 0.0010
Epoch 6/15
273/273 ━━━━━━━━━━━━━━━━━━━━ 135s 353ms/step - accuracy

In [30]:
base_model.trainable = True

In [31]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [32]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [33]:
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 158s 544ms/step - accuracy: 0.8936 - loss: 0.3256 - val_accuracy: 0.9688 - val_loss: 0.0845 - learning_rate: 1.0000e-05
Epoch 2/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9222 - loss: 0.2288
Epoch 2: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
273/273 ━━━━━━━━━━━━━━━━━━━━ 136s 497ms/step - accuracy: 0.9234 - loss: 0.2237 - val_accuracy: 0.9642 - val_loss: 0.0987 - learning_rate: 1.0000e-05
Epoch 3/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 136s 499ms/step - accuracy: 0.9351 - loss: 0.1979 - val_accuracy: 0.9679 - val_loss: 0.0943 - learning_rate: 2.0000e-06
Epoch 4/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.9276 - loss: 0.1979
Epoch 4: ReduceLROnPlateau reducing learning rate to 3.999999989900971e-07.
273/273 ━━━━━━━━━━━━━━━━━━━━ 133s 488ms/step - accuracy: 0.9301 - loss: 0.1862 - val_accuracy: 0.9670 - val_loss: 0.0917 - learning_rate: 2.0000e-06
Epoch 5/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 156s 573ms/st

In [34]:
loss, accuracy = model.evaluate(val_ds)

print("Validation Accuracy:", accuracy)

print("Validation Loss:", loss)

69/69 ━━━━━━━━━━━━━━━━━━━━ 24s 348ms/step - accuracy: 0.9688 - loss: 0.0845
Validation Accuracy: 0.9688073396682739
Validation Loss: 0.08446986228227615


In [35]:
model.save(
    "../models/efficientnet_final.keras"
)

print("Model Saved")

Model Saved


Images Shape: (32, 224, 224, 3)
Labels Shape: (32,)
Labels: [0 1 3 3 0 1 1 1 3 2]
